# Auditoria do CRM e universo preditivo

**Pergunta:** os registros públicos permitem estudar negócios ganhos/perdidos e aplicar o resultado à carteira aberta sem usar dados do futuro?

**Critério de conclusão:** conferir chaves, desfechos, campos ausentes e cronologia. Este notebook não treina um modelo. Execute a partir da pasta da submissão ou de uma subpasta; ele usa somente a biblioteca padrão do Python. O ZIP público fica em `solution/data/`, ignorado pelo Git.

In [ ]:
from __future__ import annotations
from collections import Counter
from pathlib import Path
import csv
import hashlib
import io
import shutil
import urllib.request
import zipfile

SLUG = "eduardo-celestino-de-carvalho"
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "docs" / "03-metodologia.md").is_file()), None)
if ROOT is None:
    candidate = Path.cwd() / "submissions" / SLUG
    ROOT = candidate if (candidate / "docs" / "03-metodologia.md").is_file() else None
if ROOT is None:
    raise RuntimeError("Execute a partir da pasta da submissão, de uma subpasta ou da raiz do repositório.")
DATA_DIR = ROOT / "solution" / "data"
DATA_DIR.mkdir(exist_ok=True)
print("Pasta da submissão:", ROOT)

## Plano da auditoria

1. Obter o ZIP indicado pelo challenge e registrar seu SHA-256.
2. Conferir presença das quatro tabelas e unicidade das oportunidades.
3. Separar concluídas e abertas sem rotular negócios abertos como perda.
4. Medir nulos e cobertura das junções na carteira que será priorizada.
5. Conferir datas e identificar campos conhecidos apenas depois do fechamento.

In [ ]:
SOURCE = "https://www.kaggle.com/api/v1/datasets/download/agungpambudi/crm-sales-predictive-analytics"
ZIP_PATH = DATA_DIR / "crm-sales-predictive-analytics.zip"
if not ZIP_PATH.exists():
    request = urllib.request.Request(SOURCE, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=45) as response, ZIP_PATH.open("wb") as target:
        shutil.copyfileobj(response, target)
sha256 = hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()
with zipfile.ZipFile(ZIP_PATH) as archive:
    file_names = sorted(archive.namelist())
print("Arquivo ZIP:", ZIP_PATH.name)
print("SHA-256:", sha256)
print("Arquivos:", file_names)

In [ ]:
REQUIRED = {"accounts.csv", "products.csv", "sales_teams.csv", "sales_pipeline.csv"}
assert REQUIRED.issubset(file_names), f"Tabelas ausentes: {REQUIRED - set(file_names)}"
with zipfile.ZipFile(ZIP_PATH) as archive:
    tables = {}
    for name in sorted(REQUIRED):
        with archive.open(name) as raw:
            reader = csv.DictReader(io.TextIOWrapper(raw, encoding="utf-8-sig", newline=""))
            tables[name] = list(reader)
print("Linhas por tabela:", {name: len(rows) for name, rows in tables.items()})
pipeline = tables["sales_pipeline.csv"]
ids = [row["opportunity_id"] for row in pipeline]
print("IDs de oportunidade duplicados:", len(ids) - len(set(ids)))

In [ ]:
stages = Counter(row["deal_stage"] for row in pipeline)
closed = [row for row in pipeline if row["deal_stage"] in {"Won", "Lost"}]
open_deals = [row for row in pipeline if row["deal_stage"] in {"Prospecting", "Engaging"}]
assert len(closed) + len(open_deals) == len(pipeline), "Há etapas inesperadas a investigar."
missing_accounts = {stage: sum(not row["account"].strip() for row in pipeline if row["deal_stage"] == stage) for stage in stages}
print("Etapas:", dict(stages))
print("Concluídas:", len(closed), "Abertas:", len(open_deals))
print("Sem conta por etapa:", missing_accounts)
print("Sem data de engajamento nas abertas:", sum(not row["engage_date"].strip() for row in open_deals))

In [ ]:
accounts = {row["account"] for row in tables["accounts.csv"]}
products = {row["product"] for row in tables["products.csv"]}
agents = {row["sales_agent"] for row in tables["sales_teams.csv"]}
join_audit = {
    "account_preenchida_sem_correspondencia": sum(bool(r["account"]) and r["account"] not in accounts for r in pipeline),
    "product_sem_correspondencia": sum(r["product"] not in products for r in pipeline),
    "sales_agent_sem_correspondencia": sum(r["sales_agent"] not in agents for r in pipeline),
}
print("Cobertura das junções:", join_audit)
for field in ("engage_date", "close_date"):
    values = [row[field] for row in pipeline if row[field]]
    print(field, "preenchidos", len(values), "mínimo", min(values), "máximo", max(values))
print("Campos do pipeline:", list(pipeline[0]))

In [ ]:
# Investigar falhas da junção de produto antes de normalizar.
unmatched_products = Counter(row["product"] for row in pipeline if row["product"] not in products)
print("Produtos sem correspondência:", dict(unmatched_products))
PRODUCT_ALIASES = {"GTXPro": "GTX Pro"}
assert set(unmatched_products).issubset(PRODUCT_ALIASES), "Novo produto desconhecido: revisão humana necessária."
assert set(PRODUCT_ALIASES.values()).issubset(products)
remaining = sum(PRODUCT_ALIASES.get(row["product"], row["product"]) not in products for row in pipeline)
print("Sem correspondência após normalização explícita:", remaining)


## Interpretação antes de treinar

`deal_stage` é o **desfecho final** nas linhas `Won`/`Lost`; `close_date` e `close_value` também são conhecidos só depois. Esses campos não podem entrar no modelo. Sem fotografias históricas das etapas e sem data da última interação, a avaliação retrospectiva terá limites. A próxima etapa é definir cortes temporais e uma referência simples antes de ajustar qualquer modelo.

In [ ]:
audit_summary = {
    "sha256_zip": sha256,
    "total": len(pipeline),
    "stages": dict(stages),
    "closed": len(closed),
    "open": len(open_deals),
    "open_without_account": sum(not row["account"].strip() for row in open_deals),
    "join_issues": join_audit,
}
print(audit_summary)

## Próxima decisão

- Inspecionar cronologia e disponibilidade real de cada variável candidata.
- Definir treino, validação e teste por data, com rótulos conhecidos até cada corte.
- Comparar um baseline simples com um modelo interpretável, sem usar o teste final para escolher variáveis.
- Avaliar cobertura da carteira aberta antes de prometer uma probabilidade de fechamento.